# Holiday Package Purchase Propensity

**Recruiter-facing end-to-end analysis · Binary propensity modeling · Python 3.12/3.13**

> Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.

## Executive summary

**Objective:** Estimate purchase propensity with calibrated probabilities and expose the campaign-volume precision/recall trade-off.

**Data:** 872 records with purchase outcome, salary, age, education, children, and foreign-status fields.

**Verified result:** Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.

**Decision supported:** Choose contact thresholds consistent with capacity and false-positive/false-negative costs.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A travel-campaign analyst.

**Decision:** Choose contact thresholds consistent with capacity and false-positive/false-negative costs.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '06-holiday-package-prediction'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 06-holiday-package-prediction


## 4. Data provenance and scope

Bundled in the original repository; collection population, time period, and reuse terms are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

               file  size_mb           sha256
holiday_package.csv    0.024 288f6420d7d2fbfa


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


holiday_package.csv: 8 columns
 Unnamed: 0 Holliday_Package  Salary  age  educ  no_young_children  no_older_children foreign
          1               no   48412   30     8                  1                  1      no
          2              yes   37207   45     8                  0                  1      no
          3               no   58022   46     9                  0                  0      no
          4               no   66503   31    11                  2                  0      no
          5               no   66734   44    12                  0                  2      no


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 119 lines
Functions: run_analysis


## 7. Methodology and hypotheses

Stratified CV, prevalence baseline, logistic/LDA/tree/forest/boosting comparison, probability calibration, training-only threshold selection, PR-AUC, ROC-AUC, customer profiling, and permutation importance.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_06_holiday_package_prediction", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 1.18 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (8 fields)
           column  dtype  missing_count  missing_percent  unique_values  constant
       Unnamed: 0  int64              0              0.0            872     False
 Holliday_Package object              0              0.0              2     False
           Salary  int64              0              0.0            864     False
              age  int64              0              0.0             43     False
             educ  int64              0              0.0             20     False
no_young_children  int64              0              0.0              4     False
no_older_children  int64              0              0.0              7     False
          foreign object              0              0.0              2     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'model_comparison.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.')

Primary evidence: model_comparison.csv, shape=(6, 5)
                       model  cv_pr_auc_mean  cv_pr_auc_std  cv_roc_auc_mean  cv_balanced_accuracy_mean
         logistic_regression          0.7136         0.0319           0.7240                     0.6642
           gradient_boosting          0.7096         0.0499           0.7364                     0.6611
linear_discriminant_analysis          0.7080         0.0311           0.7210                     0.6576
               random_forest          0.6988         0.0423           0.7287                     0.6729
               decision_tree          0.6364         0.0208           0.6908                     0.6291
            dummy_prevalence          0.4602         0.0034           0.5000                     0.5000

Verified result:
Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "train_rows": 654,
  "untouched_test_rows": 218,
  "selection": "5-fold stratified CV on training data",
  "selection_metric": "PR-AUC",
  "random_seed": 42
}


## 12. Visual evidence

### Holiday Propensity Evidence

![holiday_propensity_evidence](../reports/figures/holiday_propensity_evidence.png)

### Logistic Coefficients

![logistic_coefficients](../reports/figures/logistic_coefficients.png)

## 13. Business interpretation

Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.

The correct action is to use this result as evidence for **Choose contact thresholds consistent with capacity and false-positive/false-negative costs.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Use only as an educational propensity analysis; real campaigns require consent, fairness review, frequency limits, and current validation.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                       artifact  size_kb       sha256
reports\figures\holiday_propensity_evidence.png    211.8 19273084760c
      reports\figures\logistic_coefficients.png     36.8 f6f9b85ba931
                           reports\metrics.json      4.0 8ab9bc32a0d3
         reports\tables\campaign_thresholds.csv      1.1 9002c5eb55e0
 reports\tables\customer_profile_by_outcome.csv      0.2 49fe05766417
                reports\tables\data_quality.csv      0.3 23d01b326e0d
            reports\tables\model_comparison.csv      0.6 1c1bc550073f
      reports\tables\permutation_importance.csv      0.4 4b17d7cdc500


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed estimate purchase propensity with calibrated probabilities and expose the campaign-volume precision/recall trade-off. using stratified cv, prevalence baseline, logistic/lda/tree/forest/boosting comparison, probability calibration, training-only threshold selection, pr-auc, roc-auc, customer profiling, and permutation importance. The final verified conclusion is: **Calibrated logistic regression reaches untouched-test ROC-AUC 0.730 and PR-AUC 0.707; the high-recall training threshold produces 93.0% recall with low specificity.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/06-holiday-package-prediction/src/analysis.py
python scripts/execute_notebooks.py --project 06-holiday-package-prediction
```